In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [3]:
!rm -rf /content/Task-driven-low-light-enhancement
!git clone https://github.com/SarthakBaghel/Task-driven-low-light-enhancement.git /content/Task-driven-low-light-enhancement
%cd /content/Task-driven-low-light-enhancement
!pip install -r requirements.txt


Cloning into '/content/Task-driven-low-light-enhancement'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 72 (delta 15), reused 67 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 158.34 KiB | 7.20 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Task-driven-low-light-enhancement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [5]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"
CHECKPOINTS_ROOT = DRIVE_ROOT / "task_driven_checkpoints" / "kaggle_v1"

RUN_TAG = "subject_class_balanced_20k"

TEST_CLEAN_ROOT = KAGGLE_V1_ROOT / f"test_clean_{RUN_TAG}"
TEST_LOWLIGHT_ROOT = KAGGLE_V1_ROOT / f"test_lowlight_{RUN_TAG}_eye_mid"

FULL_CLEAN_CKPT_DIR = CHECKPOINTS_ROOT / f"full_clean_detector_{RUN_TAG}_resnet18"
FULL_CLEAN_BEST_CKPT = FULL_CLEAN_CKPT_DIR / f"kaggle_v1_clean_full_{RUN_TAG}_best.pt"

LOWLIGHT_CKPT_DIR = CHECKPOINTS_ROOT / f"detector_lowlight_{RUN_TAG}_eye_mid_resnet18"
LOWLIGHT_BEST_CKPT = LOWLIGHT_CKPT_DIR / "detector_lowlight_best.pt"

MAX_TOTAL_SAMPLES = 5000
SUBSET_SEED = 42

CLEAN_MODEL_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_phase8_clean_detector_{RUN_TAG}_eye_mid_5k"
LOWLIGHT_MODEL_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_phase8_lowlight_detector_{RUN_TAG}_eye_mid_5k"

assert TEST_CLEAN_ROOT.exists(), f"Missing: {TEST_CLEAN_ROOT}"
assert TEST_LOWLIGHT_ROOT.exists(), f"Missing: {TEST_LOWLIGHT_ROOT}"
assert FULL_CLEAN_BEST_CKPT.exists(), f"Missing: {FULL_CLEAN_BEST_CKPT}"
assert LOWLIGHT_BEST_CKPT.exists(), f"Missing: {LOWLIGHT_BEST_CKPT}"

print("TEST_CLEAN_ROOT:", TEST_CLEAN_ROOT)
print("TEST_LOWLIGHT_ROOT:", TEST_LOWLIGHT_ROOT)
print("FULL_CLEAN_BEST_CKPT:", FULL_CLEAN_BEST_CKPT)
print("LOWLIGHT_BEST_CKPT:", LOWLIGHT_BEST_CKPT)
print("CLEAN_MODEL_REPORT_DIR:", CLEAN_MODEL_REPORT_DIR)
print("LOWLIGHT_MODEL_REPORT_DIR:", LOWLIGHT_MODEL_REPORT_DIR)


TEST_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_clean_subject_class_balanced_20k
TEST_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_lowlight_subject_class_balanced_20k_eye_mid
FULL_CLEAN_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
LOWLIGHT_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18/detector_lowlight_best.pt
CLEAN_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_phase8_clean_detector_subject_class_balanced_20k_eye_mid_5k
LOWLIGHT_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_phase8_lowlight_detector_subject_class_balanced_20k_eye_mid_5k


In [6]:
!find "{TEST_CLEAN_ROOT}/open" -type f | wc -l
!find "{TEST_CLEAN_ROOT}/closed" -type f | wc -l

!find "{TEST_LOWLIGHT_ROOT}/open" -type f | wc -l
!find "{TEST_LOWLIGHT_ROOT}/closed" -type f | wc -l


8644
8644
8644
8644


In [7]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {FULL_CLEAN_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --lowlight-root {TEST_LOWLIGHT_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --max-total-samples {MAX_TOTAL_SAMPLES} \
  --subset-seed {SUBSET_SEED} \
  --output-dir {CLEAN_MODEL_REPORT_DIR}


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /content/Task-driven-low-light-enhancement/artifacts/torch_cache/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 197MB/s]
Clean: 100% 79/79 [24:50<00:00, 18.86s/batch]
Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Using balanced subset with 5000 samples total: {'open': 2500, 'closed': 2500}
Clean    loss: 0.1214 | accuracy: 0.9256 | precision: 0.8741 | recall: 0.9944 | f1: 0.9304 | closed_recall: 0.9944 | threshold: 0.70
Saved subset manifest to: /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/eval_phase8_clean_detector_subject_class_balanced_20k_eye_mid_5k/subset_manifest.csv
LowLight: 100% 79/79 [28:18<00:00, 21.50s/batch]
LowLight loss: 0.0410 | accuracy: 0.8960 | precision: 0.9843 | recall: 0.8048 | f1: 0.8856 | closed_recall: 0.8048 | threshold: 0.70
Saved evaluation report to: /content/drive/.shortcut-targets-by-

In [8]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {LOWLIGHT_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --lowlight-root {TEST_LOWLIGHT_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --max-total-samples {MAX_TOTAL_SAMPLES} \
  --subset-seed {SUBSET_SEED} \
  --output-dir {LOWLIGHT_MODEL_REPORT_DIR}


Clean: 100% 79/79 [02:20<00:00,  1.77s/batch]
Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Using balanced subset with 5000 samples total: {'open': 2500, 'closed': 2500}
Clean    loss: 0.3681 | accuracy: 0.5704 | precision: 0.5379 | recall: 1.0000 | f1: 0.6995 | closed_recall: 1.0000 | threshold: 0.40
Saved subset manifest to: /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/eval_phase8_lowlight_detector_subject_class_balanced_20k_eye_mid_5k/subset_manifest.csv
LowLight: 100% 79/79 [00:32<00:00,  2.43batch/s]
LowLight loss: 0.0543 | accuracy: 0.8884 | precision: 0.8228 | recall: 0.9900 | f1: 0.8987 | closed_recall: 0.9900 | threshold: 0.40
Saved evaluation report to: /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/eval_phase8_lowlight_detector_subject_class_balanced_20k_eye_mid_5k

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|--

In [9]:
from pathlib import Path
import pandas as pd

clean_manifest = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "subset_manifest.csv")
lowlight_manifest = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "subset_manifest.csv")

print("Clean model subset rows:", len(clean_manifest))
print("Low-light model subset rows:", len(lowlight_manifest))
print("Same subset:", clean_manifest.equals(lowlight_manifest))


Clean model subset rows: 5000
Low-light model subset rows: 5000
Same subset: True


In [10]:
import pandas as pd

clean_report = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "experiment_results.csv")
lowlight_report = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "experiment_results.csv")

print("=== Clean Detector ===")
print(clean_report.to_string(index=False))
print()

print("=== Low-Light Detector ===")
print(lowlight_report.to_string(index=False))


=== Clean Detector ===
  Dataset  Accuracy  Precision  Recall    F1
    Clean     92.56      87.41   99.44 93.04
Low-light     89.60      98.43   80.48 88.56

=== Low-Light Detector ===
  Dataset  Accuracy  Precision  Recall    F1
    Clean     57.04      53.79   100.0 69.95
Low-light     88.84      82.28    99.0 89.87


In [11]:
import pandas as pd

clean_report = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "experiment_results.csv")
lowlight_report = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "experiment_results.csv")

clean_report["Model"] = "Clean detector"
lowlight_report["Model"] = "Low-light detector"

combined = pd.concat([clean_report, lowlight_report], ignore_index=True)
combined = combined.rename(columns={"Dataset": "Test Domain"})

combined = combined[["Model", "Test Domain", "Accuracy", "Precision", "Recall", "F1"]]

print(combined.to_string(index=False))


             Model Test Domain  Accuracy  Precision  Recall    F1
    Clean detector       Clean     92.56      87.41   99.44 93.04
    Clean detector   Low-light     89.60      98.43   80.48 88.56
Low-light detector       Clean     57.04      53.79  100.00 69.95
Low-light detector   Low-light     88.84      82.28   99.00 89.87


In [12]:
import pandas as pd

clean_detailed = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "evaluation_results.csv")
lowlight_detailed = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "evaluation_results.csv")

clean_detailed["model"] = "Clean detector"
lowlight_detailed["model"] = "Low-light detector"

detailed = pd.concat([clean_detailed, lowlight_detailed], ignore_index=True)
display(detailed[["model", "dataset", "accuracy", "precision", "recall", "f1", "closed_recall", "threshold"]])


,model,dataset,accuracy,precision,recall,f1,closed_recall,threshold
0,Clean detector,Clean,0.9256,0.874121,0.9944,0.930389,0.9944,0.7
1,Clean detector,Low-light,0.8960,0.984344,0.8048,0.885563,0.8048,0.7
2,Low-light detector,Clean,0.5704,0.537866,1.0000,0.699496,1.0000,0.4
3,Low-light detector,Low-light,0.8884,0.822806,0.9900,0.898693,0.9900,0.4


In [13]:
def row_for(df, model_name, dataset_name):
    row = df[(df["model"] == model_name) & (df["dataset"] == dataset_name)]
    assert len(row) == 1, f"Expected one row for {model_name} / {dataset_name}"
    return row.iloc[0]

clean_on_clean = row_for(detailed, "Clean detector", "Clean")
clean_on_low = row_for(detailed, "Clean detector", "Low-light")
low_on_clean = row_for(detailed, "Low-light detector", "Clean")
low_on_low = row_for(detailed, "Low-light detector", "Low-light")

summary = pd.DataFrame([
    {
        "Question": "Low-light recovery (accuracy)",
        "Value": 100.0 * (low_on_low["accuracy"] - clean_on_low["accuracy"]),
    },
    {
        "Question": "Low-light recovery (F1)",
        "Value": 100.0 * (low_on_low["f1"] - clean_on_low["f1"]),
    },
    {
        "Question": "Low-light recovery (closed_recall)",
        "Value": 100.0 * (low_on_low["closed_recall"] - clean_on_low["closed_recall"]),
    },
    {
        "Question": "Clean retention drop/gain (accuracy)",
        "Value": 100.0 * (low_on_clean["accuracy"] - clean_on_clean["accuracy"]),
    },
    {
        "Question": "Clean retention drop/gain (F1)",
        "Value": 100.0 * (low_on_clean["f1"] - clean_on_clean["f1"]),
    },
    {
        "Question": "Clean retention drop/gain (closed_recall)",
        "Value": 100.0 * (low_on_clean["closed_recall"] - clean_on_clean["closed_recall"]),
    },
])

summary["Value"] = summary["Value"].map(lambda x: f"{x:+.2f} points")
print(summary.to_string(index=False))


                                 Question         Value
            Low-light recovery (accuracy)  -0.76 points
                  Low-light recovery (F1)  +1.31 points
       Low-light recovery (closed_recall) +18.52 points
     Clean retention drop/gain (accuracy) -35.52 points
           Clean retention drop/gain (F1) -23.09 points
Clean retention drop/gain (closed_recall)  +0.56 points


In [14]:
print("=== Clean Detector Summary ===")
print((CLEAN_MODEL_REPORT_DIR / "evaluation_summary.txt").read_text())
print()
print("=== Low-Light Detector Summary ===")
print((LOWLIGHT_MODEL_REPORT_DIR / "evaluation_summary.txt").read_text())


=== Clean Detector Summary ===
Using threshold=0.70 for the closed-eye class.
Closed-eye recall on clean data: 0.9944.
Balanced subset used for evaluation: 5000 samples total.
Per-class subset counts: {'open': 2500, 'closed': 2500}.
Closed-eye recall on low-light data: 0.8048.
Low-light degradation summary: accuracy 92.56% -> 89.60%.
Accuracy: -2.96 points (drop).
Precision: +11.02 points (change).
Recall: -18.96 points (drop).
F1: -4.48 points (drop).

=== Low-Light Detector Summary ===
Using threshold=0.40 for the closed-eye class.
Closed-eye recall on clean data: 1.0000.
Balanced subset used for evaluation: 5000 samples total.
Per-class subset counts: {'open': 2500, 'closed': 2500}.
Closed-eye recall on low-light data: 0.9900.
Low-light degradation summary: accuracy 57.04% -> 88.84%.
Accuracy: +31.80 points (change).
Precision: +28.49 points (change).
Recall: -1.00 points (drop).
F1: +19.92 points (change).


In [15]:
print("Phase 8 complete.")
print("CLEAN_MODEL_REPORT_DIR:", CLEAN_MODEL_REPORT_DIR)
print("LOWLIGHT_MODEL_REPORT_DIR:", LOWLIGHT_MODEL_REPORT_DIR)


Phase 8 complete.
CLEAN_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_phase8_clean_detector_subject_class_balanced_20k_eye_mid_5k
LOWLIGHT_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_phase8_lowlight_detector_subject_class_balanced_20k_eye_mid_5k
